# 04.5 — 密度求值与边缘积分

第 04 章通过网格积分计算边缘密度。本章把概率模型和积分算法分开讨论：

- 给定一组 $x,y$，计算 $p(x)$、$p_\theta(y|x)$ 和 $p_\theta(x,y)$。
- 给定一个 $y$，计算 $p_\theta(y)=\int p_\theta(x,y)\,\mathrm dx$。
- 给定一组 $x,y$，计算 $p_\theta(x|y)=p_\theta(x,y)/p_\theta(y)$。

前一项只需要函数求值，后两项涉及边缘积分。本章固定模型参数，研究概率计算；第 05 章再回到参数学习。

## 1. 非高斯模型：先验、条件与联合密度

定义十维隐变量 $x=(x_1,\ldots,x_d)$，各分量独立且均匀分布在 $[0,1]$ 上。先验密度为

$$p(x)=\begin{cases}1,& x\in[0,1]^d,\\0,&\text{其他}.\end{cases}$$

观测 $y\in\{0,1\}$ 服从伯努利分布：给定 $x$，发生 $y=1$ 的概率定义为

$$r(x)=\prod_{j=1}^d x_j^a,\qquad a=4.$$

因此 $p(y=1|x)=r(x)$，$p(y=0|x)=1-r(x)$。

先验是连续密度，条件是离散概率质量，二者相乘得到联合密度 $p(x,y)=p(x)p(y|x)$。这里所有公式都已知。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
d = 10
a = 4

def prior(x):
    # 独立均匀先验：判断 x 是否位于单位超立方体内。
    for value in x:
        if value < 0 or value > 1:
            return 0.0
    return 1.0

def conditional(y, x):
    # 给定 x，计算伯努利分布的参数 r(x)。
    r = 1.0
    for value in x:
        r *= value**a
    if y == 1:
        return r
    return 1 - r

def joint(x, y):
    px = prior(x)
    py_given_x = conditional(y, x)
    return px * py_given_x

# 任意给定一个十维向量与一个观测，直接计算三个概率量。
x = np.full(d, 0.8)
y = 1
print('p(x):', prior(x))
print('p(y|x):', conditional(y, x))
print('p(x,y):', joint(x, y))

## 2. 边缘密度不是一个点的联合密度

固定 $y=1$，计算

$$p(y=1)=\int_{[0,1]^d}\prod_{j=1}^d x_j^a\,\mathrm dx.$$

若沿用第 04 章的完整网格积分，每维 241 个点，十维需要 $241^{10}$ 个点。计算量来自积分方法，而非先验或条件分布的定义。

这个例子具有特殊结构：被积函数是十个独立的一维函数的乘积，因此积分可以拆开：

$$p(y=1)=\prod_{j=1}^d\int_0^1 x_j^a\,\mathrm dx_j
=(a+1)^{-d}.$$

同样可以得到 $p(y=0)=1-p(y=1)$。这里先求解析值，用它检查下一节采样方法的误差。

In [ ]:
def exact_marginal(y, d):
    p1 = (a + 1.0)**(-d)
    if y == 1:
        return p1
    return 1 - p1

print('10D grid points:', 241**d)
print('p(y=1):', exact_marginal(1, d))
print('p(y=0):', exact_marginal(0, d))

# 后验仍然使用第 02–04 章的贝叶斯公式。
posterior = joint(x, y) / exact_marginal(y, d)
print('p(x|y):', posterior)

## 3. 不遍历网格：从先验采样计算边缘密度

边缘积分也可以写成期望：

$$p(y)=\mathbb E_{x\sim p(x)}[p(y|x)].$$

从先验独立采样 $M$ 次，使用条件概率的样本均值估计：

$$\widehat p_M(y)=\frac{1}{M}\sum_{j=1}^M p(y|x^{(j)}).$$

只需要 $M$ 次条件概率求值。先验的信息已包含在采样过程里，不需要显式计算 $p(x)$ 或联合密度。

In [ ]:
def sample_marginal(y, d, M, conditional_fn):
    total = 0.0
    for j in range(M):
        # 从 d 维独立均匀先验中采样。
        x = rng.uniform(0, 1, size=d)
        total += conditional_fn(y, x)
    return total / M

for dim in [1, 2, 10]:
    exact = exact_marginal(1, dim)
    estimate = sample_marginal(1, dim, 1000, conditional)
    relative_error = (estimate - exact) / exact
    print('dimension:', dim)
    print('exact:', exact, 'MC:', estimate)
    print('relative error:', relative_error)

## 4. 可以采样，不等于少量采样就足够

对同一个模型、同一个观测，多次独立重复积分。比较低维与高维情况下估计值的波动，以及常见观测 $y=0$ 和罕见观测 $y=1$ 的区别。

估计值除以解析值，理想结果为 1。

In [ ]:
M = 1000
repeats = 20

for dim in [1, 10]:
    for y in [0, 1]:
        exact = exact_marginal(y, dim)
        ratios = []
        for run in range(repeats):
            estimate = sample_marginal(y, dim, M, conditional)
            ratios.append(estimate / exact)
        print('dimension:', dim, 'y:', y)
        print('estimate/exact: mean =', np.mean(ratios),
              'std =', np.std(ratios))
        plt.plot(ratios, 'o-', label=f'd={dim}, y={y}')

plt.axhline(1.0, color='black', linestyle='--', label='exact')
plt.xlabel('independent repeat')
plt.ylabel('estimated marginal / exact marginal')
plt.legend()
plt.show()

还可以直接计算这个例子的采样误差。令 $Z=p(y=1|x)$，其中 $x\sim p(x)$。由于各分量独立，

$$\mathbb E[Z]=(a+1)^{-d},\qquad
\mathbb E[Z^2]=(2a+1)^{-d}.$$

独立采样 $M$ 次后，样本均值的标准差除以真实边缘概率为

$$\frac{\sqrt{\operatorname{Var}(\widehat p_M)}}{p(y=1)}
=\sqrt{\frac{((a+1)^2/(2a+1))^d-1}{M}}.$$

网格点数量的指数增长被绕开了，但为了维持相同的相对误差，这个模型所需的采样数量仍随维度指数增长。这是本例的性质，不是所有蒙特卡洛积分的普遍规律。

In [ ]:
for dim in [1, 2, 10]:
    relative_std = np.sqrt((((a+1)**2/(2*a+1))**dim - 1) / M)
    print('dimension:', dim, 'relative std:', relative_std)

## 5. 去掉可分离结构

前面的伯努利参数是独立分量的乘积，所以边缘积分有解析解。现在只修改条件分布，在原来的 $r(x)$ 上乘一个依赖所有分量的因子：

$$r_c(x)=r(x)\left[0.5+0.4\sin\left(8\left(\sum_{j=1}^d x_j\right)^2\right)\right].$$

括号中的值始终在 $[0.1,0.9]$，因此仍是合法的伯努利参数。先验不变，仍然有 $p_c(x,y)=p(x)p_c(y|x)$。

逐点计算依然简单，但原来将积分拆为十个一维积分的公式不再适用。我们可以继续使用同一个采样函数。

In [ ]:
def conditional_coupled(y, x):
    r = conditional(1, x)
    factor = 0.5 + 0.4 * np.sin(8 * np.sum(x)**2)
    p1 = r * factor
    if y == 1:
        return p1
    return 1 - p1

def joint_coupled(x, y):
    return prior(x) * conditional_coupled(y, x)

x = np.full(d, 0.8)
y = 1
print('prior:', prior(x))
print('conditional:', conditional_coupled(y, x))
print('joint:', joint_coupled(x, y))

# 没有使用解析参考值；重复估计，观察不同采样数量下的波动。
for M in [100, 1000, 10000]:
    estimates = []
    for run in range(5):
        estimate = sample_marginal(y, d, M, conditional_coupled)
        estimates.append(estimate)
    print('M:', M, 'estimates:', estimates)

## 6. 与后验和 MLE 的关系

对于上面的耦合模型，贝叶斯公式仍然成立：

$$p_c(x|y)=\frac{p(x)p_c(y|x)}{\int p(x')p_c(y|x')\,\mathrm dx'}.$$

分子可以直接计算，分母需要积分。分子只是一个关于 $x$ 的未归一化后验函数。知道它的表达式，并不等于知道归一化后的后验密度。

如果模型参数未知，MLE 仍然是最大化 $\sum_i\log p_\theta(y_i)$。可以用先验采样估计每个边缘概率，但有限样本下，即使 $\widehat p_M(y)$ 是无偏估计，$\log\widehat p_M(y)$ 一般也有偏；估计值的波动还会影响训练。

第 05 章回到第 04 章的神经网络生成模型。它引入依赖观测 $y$ 的近似后验 $q_\phi(x|y)$，并构造 ELBO 作为训练目标。这里不要求先求出真实后验，也不要求遍历隐变量空间。